In [194]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [195]:
from typing import Literal

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, util
from datasets import Dataset

from src.preprocessor.tree_preprocessor import TreePreprocessor
from src.paths import DATA_DIR
from src.parser.trees import Tree
from src.Qlassifier.baseline import load_data, get_paragraphs

# Setting up data

In [196]:
def collapse_doc(
    root: Tree,
    doc_type: Literal["past_exam", "study_design"]
):
    if doc_type == "past_exam":
        level = root.find_node_level(r"Question")
    else: 
        level = root.find_node_level(r"Outcome \d")
    root.collapse(level)

In [197]:
subjects = [
    "chemistry"
]
subject = subjects[0]
exam = "2023"
exam_dir = DATA_DIR / subject / "past_exams"
exam_path = exam_dir / f"{exam}.pdf"

In [198]:
def prepare_dataframes(exam_path: str):

    ex_root, report_df, sd_root = load_data(exam_path)
    sd_root = TreePreprocessor("study_design", remove_latex=True).preprocess(sd_root, subject=subject)
    ex_root = TreePreprocessor("past_exam", remove_latex=True).preprocess(ex_root)

    collapse_doc(ex_root, "past_exam")
    ex_df, sd_df = ex_root.to_df(include_root=False), sd_root.to_df(include_root=False)

    sd_df = sd_df.loc[
        ~(sd_df["label"].str.contains(
            r"(Unit \d)|(Area of Study \d)|(Key Knowledge)|(Outcome \d)", case=False, na=False
        ) | sd_df["text"].str.contains("In this area of study"))
    ].reset_index(drop=True)
    report_comments = [df[["comments"]] for df in report_df]

    comments_df = pd.concat(report_comments).reset_index(drop=True)

    ex_df = ex_df.loc[ex_df["label"].str.contains("Question", na=False), :].reset_index(drop=True)
    return ex_df, comments_df, sd_df

ex_df, comments_df, sd_df = prepare_dataframes(exam_path)

/tmp/ipykernel_100379/3283292783.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~(sd_df["label"].str.contains(


In [204]:
sd_df.head(5)

,label,text
0,Carbon-based fuels,"the definition of a fuel, including the disti..."
1,Measuring changes in chemical reactions,calculations related to the application of st...
2,Primary galvanic cells and fuel cells as sourc...,redox reactions as simultaneous oxidation and...
3,Rates of chemical reactions,factors affecting the frequency and success o...
4,Extent of chemical reactions,the distinction between reversible and irreve...


In [203]:
ex_df.head(5)

,label,text
0,Question 1,Which one of the following correctly represent...
1,Question 2,Which one of the following statements is corre...
2,Question 3,"Molecules X, and all have the same number of c..."
3,Question 4,cell can be considered secondary cell if A. it...
4,Question 5,Consider the following statements about coenzy...


In [202]:
comments_df.head(5)

,comments
0,"In human cells, glucose reacts with oxygen in ..."
1,Fuel cells and galvanic cells both produce hea...
2,The larger the number of C=C double bonds in t...
3,The polarity of the physical electrodes does n...
4,All three statements are properties of coenzym...


# Setting up Model (without fine-tuning)

In [116]:
sd_df["input"] = sd_df["label"].str.cat(others=sd_df["text"], sep=": ")
ex_df["input"] = ex_df["text"]

In [139]:
model = SentenceTransformer("intfloat/e5-base-v2")

In [ ]:
qn_input = list("query:" + ex_df["text"])
report_input = list("query:" + comments_df["comments"]) 
sd_input = list("passage:" + sd_df["input"])

sd_emb = model.encode(sd_input, convert_to_tensor=True, normalize_embeddings=True)
report_emb = model.encode(report_input, convert_to_tensor=True, normalize_embeddings=True)
qn_emb = model.encode(qn_input, convert_to_tensor=True, normalize_embeddings=True)

In [ ]:
cos = util.cos_sim(qn_emb, sd_emb) # Embedding similarity b/w actual exam questions and study design
best_idxs = [torch.argmax(sims).item() for sims in cos]

cos_qn = util.cos_sim(report_emb, sd_emb) # Embedding similarity b/w REPORT section on the question and study design
best_rp_idxs = [torch.argmax(sims).item() for sims in cos_qn]

In [ ]:
ex_df["predicted_topic"] = sd_df.loc[best_idxs, "label"].reset_index(drop=True)
ex_df["predicted_topic_idx"] = best_idxs # exam's prediction
ex_df["report_predict"] = best_rp_idxs # report's prediction

## Compare against baseline!

In [ ]:
true_topic_indices = [
    0, 2, 6, 5, 10, 9, 0, 4, 0, 4,
    3, 3, 6, 4, 0, 9, 1, 9, 6, 8,
    10 ,2, 6, 2, 5, 11, 12, 2, 9, 2,
]
mcq_df = ex_df.loc[:29, :].copy()
mcq_df["true_topic_idx"] = true_topic_indices
mcq_df["true_topic"] = sd_df.loc[true_topic_indices, "label"].reset_index(drop=True)

In [187]:
mcq_df.head(5)

,label,text,predicted_topic,predicted_topic_idx,input,report_predict,true_topic_idx,true_topic
0,Question 1,Which one of the following correctly represent...,Carbon-based fuels,0,Which one of the following correctly represent...,0,0,Carbon-based fuels
1,Question 2,Which one of the following statements is corre...,Primary galvanic cells and fuel cells as sourc...,2,Which one of the following statements is corre...,2,2,Primary galvanic cells and fuel cells as sourc...
2,Question 3,"Molecules X, and all have the same number of c...","Structure, nomenclature and properties of orga...",6,"Molecules X, and all have the same number of c...",6,6,"Structure, nomenclature and properties of orga..."
3,Question 4,cell can be considered secondary cell if A. it...,Primary galvanic cells and fuel cells as sourc...,2,cell can be considered secondary cell if A. it...,2,5,Production of chemicals using electrolysis
4,Question 5,Consider the following statements about coenzy...,Reactions of organic compounds,7,Consider the following statements about coenzy...,7,10,Medicinal chemistry


In [189]:
ex_correct_pc = mcq_df[mcq_df["true_topic_idx"] == mcq_df["predicted_topic_idx"]].shape[0] / mcq_df.shape[0]
rp_correct_pc = mcq_df[mcq_df["true_topic_idx"] == mcq_df["report_predict"]].shape[0] / mcq_df.shape[0]

one_correct_pc = mcq_df[(mcq_df["true_topic_idx"] == mcq_df["predicted_topic_idx"]) |
       (mcq_df["true_topic_idx"] == mcq_df["report_predict"])].shape[0] \
/ mcq_df.shape[0]

print(f"Exam-only accuracy: {100*ex_correct_pc:.2f}%")
print(f"Report-only accuracy: {100*rp_correct_pc:.2f}%")
print(f"The percent of time either the report or the exam labels a question correctly is {100*one_correct_pc:.2f}%")

Exam-only accuracy: 63.33%
Report-only accuracy: 63.33%
The percent of time either the report or the exam labels a question correctly is 76.67%


Perhaps we can combine the two model's predictions in some way to models accuracy? Probably not worth it given either of them are correct only 77% of the time, barely better than our baseline. 

The fun part! Optimising the model's performance. 

# Fine-tuning

We a lot of data if we want to make a meaningful change in the model's understanding of the context. Let's use all data up until the target examinations years (2023, for now).

In [ ]:
training_exam_dfs = []
training_report_dfs = []

for exam_path in exam_dir.iterdir():
    if exam in str(exam_path):
        # Don't train on current dataset 
        continue
    curr_exam = exam_path.stem
    train_exam_df, train_report_df, _ = prepare_dataframes(exam_path)
    train_exam_df.insert(0, "Exam Year", [curr_exam]*len(train_exam_df))
    train_report_df.insert(0, "Exam Year", [curr_exam]*len(train_report_df))

    training_exam_dfs.append(train_exam_df)
    training_report_dfs.append(train_report_df)

In [ ]:
# ["Exam Year", "Label"] unique for each year
training_ex_df = pd.concat(training_exam_dfs) \
                   .reset_index(drop=True) \
                   .drop_duplicates(subset=["Exam Year", "label"])

training_rp_df = pd.concat(training_report_dfs) \
                   .reset_index(drop=True) \
                   .drop_duplicates(subset=["Exam Year", "label"]) 

In [271]:
training_ex_df

,Exam Year,label,text
0,2018,Question 1,Which one of the following statements is the m...
1,2018,Question 2,Aspartame is widely used sweetener. Aspartame ...
2,2018,Question 3,Which one of the following statements about fu...
3,2018,Question 4,"At the molecular level, Protein is shaped like..."
4,2018,Question 5,Coal seam gas is used to generate electricity....
...,...,...,...
476,2019,Question 9f.,student designed an experiment to investigate ...
477,2019,Question 9g.,student designed an experiment to investigate ...
478,2019,Question 9h.,student designed an experiment to investigate ...
479,2019,Question 10a.,Climate change has been identified as threat t...
